# Delete job: removes a deleted app job's groups and, if asked, their target tables

The operations app has no DELETE privilege on the Lakebase tables. When an ingestion job is deleted in the app, it
marks the job deleted and its groups removed, then starts this job, which runs as the deployer (the state database's
owner) and finishes the delete (`delete_job.py`):

- **Rows:** in one transaction, under each group's run lock, the groups' control rows and their checkpoint,
  reseed_queue, database_check and check_work rows. Groups another live job holds stay. A run of any group refuses
  the lot, and nothing is deleted.
- **Kept:** run history (ingest_run, ingest_event) and the change log (control_change).
- **Target tables** (`drop_targets` true): after the commit, `DROP TABLE IF EXISTS` for each table only these groups
  wrote. A table another group's rows also write is kept and named. A schema is never dropped.
- **Drafts** (`purge_drafts` true, no `schedule_id`): the tables the app read for a choice no job took (discarded,
  or not asked about for `draft_ttl_s`) are purged: their discovery and discovered_table rows, in one transaction.
  The app starts this at most hourly, and only when there is a draft to purge.

In [ ]:
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

from state_store import StateStore
import delete_job


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


schedule_id = widget("schedule_id")
drop_targets = widget("drop_targets", "false").lower() in ("1", "true", "yes")
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
purge_drafts = widget("purge_drafts", "false").lower() in ("1", "true", "yes")
draft_ttl_s = int(widget("draft_ttl_s", str(delete_job.DRAFT_TTL_S)) or delete_job.DRAFT_TTL_S)
assert purge_drafts or schedule_id.isdigit(), "schedule_id is required: the deleted ingestion job's id"

In [ ]:
store = StateStore.connect_lakebase(db_instance, db_name)
if purge_drafts:
    # The drafts mode does only that: no job is deleted and no table dropped.
    try:
        with store.connection.cursor() as cur, cur.connection.transaction():
            purged = delete_job.purge_drafts(cur, draft_ttl_s)
    finally:
        store.close()
    print(f"purged {purged['drafts']} draft(s): {purged['databases']} discovery row(s), {purged['tables']} table(s)")
    dbutils.notebook.exit(json.dumps({"purged": purged}))
try:
    with store.connection.cursor() as cur, cur.connection.transaction():
        result = delete_job.delete_rows(cur, int(schedule_id))
finally:
    store.close()
print(f"ingestion job {result['job']} ({schedule_id}): {len(result['groups'])} group(s) "
      f"{', '.join(result['groups']) or '-'}; deleted {json.dumps(result['deleted'])}")
kept = result["targets"]["kept"]
if kept:
    print(f"{len(kept)} target table(s) kept, since another group's rows also write them: "
          + ", ".join(kept[:delete_job.LISTED]) + (" and more" if len(kept) > delete_job.LISTED else ""))

In [ ]:
# After the commit: a drop can't be undone, so the rows go first. A failed drop doesn't stop the others; the run
# fails at the end naming each one, since the rows that listed it are gone and a second run won't find it.
# Several at a time: a job of many thousands of tables would take hours one by one.
from concurrent.futures import ThreadPoolExecutor


def drop(table):
    try:
        spark.sql(delete_job.drop_sql(table))
        return table, None
    except Exception as e:
        return table, str(e)[:300]


dropped, failed = [], {}
if drop_targets:
    with ThreadPoolExecutor(max_workers=16) as pool:
        for table, error in pool.map(drop, result["targets"]["drop"]):
            if error:
                failed[table] = error
            else:
                dropped.append(table)
    print(f"dropped {len(dropped)} target table(s)" + (f", {len(failed)} failed" if failed else ""))
else:
    print(f"target tables kept ({len(result['targets']['drop'])} only this job wrote)")
summary = delete_job.summary(result, dropped, failed)
print(json.dumps(summary, indent=2))
if failed:
    for table, error in failed.items():
        print(f"not dropped: {table}: {error}")
    raise RuntimeError(delete_job.failure_message(failed))
dbutils.notebook.exit(json.dumps(summary))